Silver Layer --> Cleaning and Structuring Data

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Defining Spark session
spark = SparkSession.builder.appName("JobETL_Silver").getOrCreate()

# Reading Bronze layer
print("Reading from Bronze Layer -------> Silver Layer")

silver_df = spark.table("job_market.bronze.raw_jobs")

silver_df = silver_df.selectExpr(
    "title",
    "company_name",
    "location",
    "description",
    "job_id",
    "extensions",
    "search_role"
)

# Extract posting date from any position in the extensions array
silver_df = silver_df.withColumn(
    "posted_at",
    get(
        filter(
            col("extensions"),
            lambda x: x.rlike(
                r"(?i).*(posted|ago|today|yesterday).*"
            )
        ),
        0
    )
)

# Keep only rows with required fields
silver_df = silver_df.dropna(
    subset=["title", "company_name", "location"]
)

# Select final Silver layer columns
silver_df = silver_df.select(
    "title",
    "company_name",
    "location",
    "description",
    "job_id",
    "posted_at",
    "search_role"
)

display(silver_df)

In [0]:
silver_df = silver_df.withColumn('company_name', trim(upper(col('company_name'))))
silver_df = silver_df.dropna(subset = ['posted_at'])

# save data as delta table
silver_df.write.format('delta').option('mergeSchema', 'true').mode('append').saveAsTable('job_market.silver.jobs')
print("✅ Data Written to Silver Layer!")
# display(silver_df)

In [0]:
%sql
SELECT * FROM job_market.silver.jobs;